In [6]:
import pandas as pd

import sys
sys.path.append("..")
from src.strategies import run_momentum_strategy
from src.backtester import calculate_return

df = pd.read_csv("../data/spy_daily.csv")
prices = df["Adj Close"]
returns = calculate_return(prices)
lookback = 252
transaction_cost = 0.001

In [7]:
_, momentum_results = run_momentum_strategy(prices, returns, lookback, transaction_cost)
print(pd.Series(momentum_results))

total_return          5.400420
cagr                  0.075102
annual_volatility     0.128810
annual_sharpe         0.627024
max_drawdown         -0.315812
calmar                0.237805
time_invested         0.775232
trades               89.000000
dtype: float64


In [8]:
df1 = pd.read_csv("../data/elementary_random_strategy_summary.csv")
df2 = pd.read_csv("../data/exposure_matched_random_strategy_summary.csv")
df3 = pd.read_csv("../data/trade_count_matched_random_strategy_summary.csv")
df4 = pd.read_csv("../data/exposure_+_trade_count_matched_random_strategy_summary.csv")
df5 = pd.read_csv("../data/run_length_permutation_random_strategy_summary.csv")
df6 = pd.read_csv("../data/cyclic_shift_random_strategy_summary.csv")


In [9]:
frames = [df1, df2, df3, df4, df5, df6]

results = pd.concat(frames, ignore_index = True)

results.columns = results.columns.str.title().str.replace("_", " ")

results["Strategy"] = (
    results["Strategy"]
    .str.replace("_random_strategy", "")
    .str.replace("_", " ")
    .str.title()
)

results

,Strategy,Mean Sharpe,Median Sharpe,Max Sharpe,P Value,Avg Trades,Avg Time Invested,Number Of Simulations
0,Elementary,-0.542481,-0.542160,0.092003,0.000010,3229.84590,0.500002,100000
1,Exposure Matched,-0.036150,-0.036469,0.380730,0.000010,2251.87858,0.775208,100000
2,Trade Count Matched,0.375792,0.370519,1.010502,0.036490,89.00000,0.499999,100000
3,Exposure + Trade Count Matched,0.469425,0.464082,0.946090,0.047090,89.00000,0.775232,100000
4,Run Length Permutation,0.473815,0.460054,0.984638,0.077839,89.00000,0.775232,100000
5,Cyclic Shift,0.473505,0.442368,1.181052,0.110169,90.76216,0.775232,100000


In [10]:
table = results.copy()

table = table.round({
    "Mean Sharpe": 4,
    "Median Sharpe": 4,
    "Max Sharpe": 4,
    "P Value": 5,
    "Avg Trades": 2,
    })

table["Avg Time Invested"] = table["Avg Time Invested"].map(
    lambda x: f"{x * 100:.2f}%"
)

table

,Strategy,Mean Sharpe,Median Sharpe,Max Sharpe,P Value,Avg Trades,Avg Time Invested,Number Of Simulations
0,Elementary,-0.5425,-0.5422,0.0920,0.00001,3229.85,50.00%,100000
1,Exposure Matched,-0.0361,-0.0365,0.3807,0.00001,2251.88,77.52%,100000
2,Trade Count Matched,0.3758,0.3705,1.0105,0.03649,89.00,50.00%,100000
3,Exposure + Trade Count Matched,0.4694,0.4641,0.9461,0.04709,89.00,77.52%,100000
4,Run Length Permutation,0.4738,0.4601,0.9846,0.07784,89.00,77.52%,100000
5,Cyclic Shift,0.4735,0.4424,1.1811,0.11017,90.76,77.52%,100000


## Interpretation

#### 

The momentum strategy being benchmarked returned an observed annual Sharpe ratio of approx. 0.627. The strategy was invested in the SPY around 77.52% of the time, with 89 trades occuring across the time period. 



1. Elementary Random Benchmark - $H_0$(1): Momentum performs no different to an independently random 50/50 market-timing strategy on a risk-adjusted basis. 

    In the Elementary random strategy, each valid trading day has a 50% chance of being in an invested/cash state. 

    Momentum substantially outperforms this simplistic benchmark on a risk-adjusted basis; in fact, Momentum's annual Sharpe ratio exceeds the maximum Sharpe observed across all 100,000 simulated strategies by 0.5350. This is reflected in an empirical p-value of 0.00001, providing very strong evidence against $H_0$(1). 


2. Exposure-Matched Random Benchmark - $H_0$(2): Momentum performs no different to an independently random exposure-matched market-timing strategy on a risk-adjusted basis. 

    The Exposure-Matched random strategy aims to take comparable amounts of market risk to Momentum for a more representative benchmark. Each valid trading day has an approx. 77.52% chance of being in an invested state. Indeed, the average time invested across the Exposure-Matched simulations ends up at the 77.52% mark. 

    Momentum exhibits significantly stronger performance returns relative to risk, exceeding the mean Exposure-Matched Sharpe ratio by 0.6631. Again, with an empirical p-value of 0.00001, there is very strong evidence to reject $H_0$(2). 


3. Trade-Count Matched Random Benchmark - $H_0$(3): Momentum performs no different to a strategy with 89 randomly selected trade positions on a risk-adjusted basis. REPHRASE MAYBE

    Strategies 1. and 2. are punished by the large volume of trades occuring and the transaction costs these trades incur. This benchmark aims to resolve this by matching the number of trades occuring in Momentum (89). As the position of the trade days are independently randomly selected, split between time invested/time in cash is approx. 50/50. 

    Momentum achieved materially stronger risk-adjusted performance than the benchmark, outperforming the mean Sharpe ratio by 0.2512. A p-value of 0.03649 provides evidence against the null, so we reject $H_0$(3) at the 5% significance level. 

    Consider that the Trade-Count Matched strategy performs considerably better than previous strategies, suggesting the large trade count prior had a significant negative impact on risk-adjusted performance. 


4. Exposure + Trade-Count Matched Random Benchmark - $H_0$(4): Momentum performs no different to a strategy with 89 randomly selected trade positions and exposure-matched market-timing on a risk-adjusted basis.

    This benchmark aims to resemble the Momentum signal, by matching both trade count and exposure in the same random strategy. This is demonstrated, with an average time invested across 100,000 simulations as 77.52%, and an average trade count of 89. 

    Momentum Sharpe exceeds the benchmark distribution by a meaninful margin, with p = 0.04709 < 0.05>. Thus, there is modest evidence to suggest a difference in risk-adjusted performance between momentum and a trade-count/exposure-matched random benchmark (reject $H_0$(4) at 5% significance level). Notably, evidence against $H_0$ has weakened consistently as the random benchmarks more closely resemble Momentum's structure. 


5. Run Length Permutation Random Benchmark - $H_0$(5): Momentum peforms no different to random permutations of its cash and invested run lengths. 

    This random strategy preserves the lengths of Momentum's cash/invested runs, and randomly reorders them, producing signals with similar structure to Momentum, and identical exposure/time invested. Indeed, this signal resembles Momentum's more closely than any strategy prior. 

    Momentum's Sharpe exceeds the average run length permutation by 0.1532. However, an empirical p-value of 0.07784 fails to reject the null hypothesis at the 5% significance level. There is insufficient evidence to conclude that momentum outperforms a random permutation of its own cash and invested runs. 


6. Cyclic Shifting Random Benchmark - $H_0$(6): Momentum performs no different than the same trading pattern shifed to different points in market history. 

    This random strategy preserves the structure of Momentum, but shifts the signal forward by a random amount, changing which market returns the positions line up with. This replicates the frequent flutters between buy/sell that the run length permutation benchmark would often fail to. 

    A p-value of 0.11017 is obtained, thus we fail to reject $H_0$(6) at the 5% signficance level. There is insufficient evidence to suggest that momentum outperforms a cyclic shift of its own trading signal through market history. 